In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt

In [2]:
df = pd.read_excel('RFL_feature.xlsx')

In [3]:
df

,Frequency,Particulate,RelativeHumidity,Visibility,AbsoluteHumidityMax,ParticulateMax,RainIntensityMax,Temperature,RainIntensityMin,AbsoluteHumidity,RainIntensity,Distance,RFL_Att
0,73500000000,0.000000,80.291130,50251.391340,17.504117,0.000000,0.000000,22.384038,0.000000,15.930662,0.000000,2959.931772,9.694687
1,73500000000,0.000000,36.754752,19554.715500,5.139645,0.000000,0.000000,15.593858,0.000000,4.884168,0.000000,2018.767588,11.336494
2,73500000000,0.000000,81.129278,15287.325520,22.967831,0.000000,0.000000,27.592859,0.000000,21.586978,0.000000,2956.858380,10.266708
3,73500000000,0.000000,70.212589,25159.573780,4.186823,0.000000,0.000000,2.632763,0.000000,4.074320,0.000000,4818.176068,11.443969
4,73500000000,0.000000,68.977211,30345.736030,19.071630,0.000000,0.000000,27.080075,0.000000,17.840132,0.000000,2955.199128,8.726055
...,...,...,...,...,...,...,...,...,...,...,...,...,...
42859,83500000000,16.957986,76.290310,27589.122520,5.841524,18.215495,0.058729,5.722957,0.052778,5.437567,0.058399,4825.841905,13.686893
42860,73500000000,27.278849,95.759465,12247.857120,20.202120,28.111408,0.000000,22.057799,0.000000,18.646348,0.000000,2115.649872,5.726912
42861,73500000000,174.912616,88.914893,7879.930898,11.686584,188.515642,0.000000,14.982220,0.000000,11.384310,0.000000,2018.153150,13.166719
42862,73500000000,5.954184,76.946078,74106.624310,6.182977,5.964829,0.003118,6.939353,0.002844,5.938114,0.003016,4819.435242,13.244202


In [4]:
X = df.drop(columns=["RFL_Att"])
y = df["RFL_Att"]

In [5]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [6]:
param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [10, None],
    'min_samples_split': [2],
    'max_features': ['sqrt']
}

In [7]:
grid_search = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_grid=param_grid,
    scoring='neg_root_mean_squared_error',
    cv=5,
    n_jobs=-1,
    verbose=1
)

In [8]:
grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 4 candidates, totalling 20 fits


GridSearchCV(cv=5, estimator=RandomForestRegressor(random_state=42), n_jobs=-1,
             param_grid={'max_depth': [10, None], 'max_features': ['sqrt'],
                         'min_samples_split': [2], 'n_estimators': [50, 100]},
             scoring='neg_root_mean_squared_error', verbose=1)

In [9]:
print("Best Parameters:", grid_search.best_params_)
print("Best CV Score (Negative RMSE):", grid_search.best_score_)

Best Parameters: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_split': 2, 'n_estimators': 100}
Best CV Score (Negative RMSE): -0.6921632907642616


In [10]:
best_model = grid_search.best_estimator_
best_model.fit(X_train, y_train)

RandomForestRegressor(max_features='sqrt', random_state=42)

In [11]:
y_train_pred = best_model.predict(X_train)
y_val_pred = best_model.predict(X_val)

In [12]:
rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
rmse_val = np.sqrt(mean_squared_error(y_val, y_val_pred))

In [13]:
r2_train = r2_score(y_train, y_train_pred)
r2_val = r2_score(y_val, y_val_pred)

In [14]:
print(f"Training RMSE: {rmse_train:.4f}, Validation RMSE: {rmse_val:.4f}")
print(f"Training R²: {r2_train:.4f}, Validation R²: {r2_val:.4f}")

Training RMSE: 0.2613, Validation RMSE: 0.7017
Training R²: 0.9939, Validation R²: 0.9556


## Adding predicted RF

In [49]:


train_results = pd.DataFrame({
    "Original_RFL_Att": y_train.values,
    "Predicted_RFL_Att": y_train_pred
})
val_results = pd.DataFrame({
    "Original_RFL_Att": y_val.values,
    "Predicted_RFL_Att": y_val_pred
})

# Combine predictions
all_results = pd.concat([train_results, val_results])

# Merge predictions back on matching actual RFL_Att (retain original untouched)

train_original = pd.read_excel('train_dataset.xlsx')


In [51]:
train_original["Predicted_RFL_Att"] = None

for i in train_original.index:
    match = all_results[all_results["Original_RFL_Att"] == train_original.loc[i, "RFL_Att"]]
    if not match.empty:
        train_original.loc[i, "Predicted_RFL_Att"] = match.iloc[0]["Predicted_RFL_Att"]

In [53]:
train_original.to_excel("RFL_pred_dataset.xlsx", index=False)

## Test the model

In [35]:
test_df = pd.read_excel('test_dataset.xlsx')

In [22]:
selected_features = [
    'Frequency', 'Particulate', 'RelativeHumidity', 'Visibility', 'AbsoluteHumidityMax',
    'ParticulateMax', 'RainIntensityMax', 'Temperature', 'RainIntensityMin',
    'AbsoluteHumidity', 'RainIntensity', 'Distance','RFL_Att'
]

# Filter only the selected features (if they exist in the dataset)
filtered_test_df = test_df[[col for col in selected_features if col in test_df.columns]]

In [16]:
#test_df

In [3]:
X_test = filtered_test_df.drop(columns=["RFL_Att"])
y_test = filtered_test_df["RFL_Att"]

NameError: name 'filtered_test_df' is not defined

In [26]:
y_test_pred = best_model.predict(X_test)

In [28]:
rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))
r2_test = r2_score(y_test, y_test_pred)

In [30]:
print(f"Testing RMSE: {rmse_test:.4f}")
print(f"Testing R²: {r2_test:.4f}")

Testing RMSE: 0.7156
Testing R²: 0.9563


In [60]:
test_df["Predicted_RFL_Att"] = y_test_pred

# Save to a new Excel file
test_df.to_excel("test_RFL_dataset_with_predictions.xlsx", index=False)